# LeetCode #1420: Build Array Where You Can Find The Maximum Exactly K Comparisons

https://leetcode.com/problems/build-array-where-you-can-find-the-maximum-exactly-k-comparisons/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(m^n)$ | $O(n)$ |
| **Optimal: 3D DP ★** | $O(n \cdot k \cdot m^2)$ | $O(n \cdot k \cdot m)$ |

---

## Understanding the Methods

### Brute Force
Enumerate every array of length `n` with values in `[1, maxValue]`, simulate the search algorithm, and count those with exactly `k` comparisons. Exponential — unusable even for small `n`.

### Optimal: 3D DP ★
`dp[i][j][c]` = number of ways to fill positions `0..i` such that the current maximum is `j` and exactly `c` new-max comparisons have been made. Transition: either place a value $\leq j$ (no new comparison) or place a value $v > j$ (one new comparison, new max = `v`). Process left to right and accumulate with prefix sums to speed up the "place smaller" transitions.

**Why this is better than Brute Force:** We reuse overlapping sub-problems; the 3D table has $O(n \cdot k \cdot m)$ states, each costing $O(m)$ transitions — polynomial in all three parameters.

**Constraints:**
* $1 \leq n \leq 50$
* $1 \leq m \leq 100$ (`maxValue`)
* $1 \leq k \leq n$

## Solutions

### C#

In [ ]:
public class Solution {
    public int NumOfArrays(int n, int m, int k) {
        const int MOD = 1_000_000_007;
        // dp[pos][max_so_far][comparisons]
        // = ways to fill positions 0..pos with current max = max_so_far and c comparisons used
        var dp = new long[n + 1, m + 1, k + 1];
        // Base: empty array, max = 0, 0 comparisons — 1 way
        dp[0, 0, 0] = 1;

        for (int pos = 0; pos < n; pos++) {
            for (int j = 0; j <= m; j++) {
                for (int c = 0; c <= k; c++) {
                    if (dp[pos, j, c] == 0) continue;
                    // Place a value <= j (no new comparison)
                    for (int v = 1; v <= j; v++)
                        dp[pos + 1, j, c] = (dp[pos + 1, j, c] + dp[pos, j, c]) % MOD;
                    // Place a value > j (new comparison, new max = v)
                    if (c + 1 <= k)
                        for (int v = j + 1; v <= m; v++)
                            dp[pos + 1, v, c + 1] = (dp[pos + 1, v, c + 1] + dp[pos, j, c]) % MOD;
                }
            }
        }

        long ans = 0;
        for (int j = 1; j <= m; j++)
            ans = (ans + dp[n, j, k]) % MOD;
        return (int)ans;
    }
}

### Python

In [ ]:
class Solution:
    def num_of_arrays(self, n: int, m: int, k: int) -> int:
        MOD = 10**9 + 7
        # dp[pos][max_so_far][comparisons]
        # = ways to fill positions 0..pos with current max = max_so_far and c comparisons
        dp = [[[0] * (k + 1) for _ in range(m + 1)] for _ in range(n + 1)]
        # Base: empty array, max = 0, 0 comparisons — 1 way
        dp[0][0][0] = 1

        for pos in range(n):
            for j in range(m + 1):
                for c in range(k + 1):
                    if not dp[pos][j][c]:
                        continue
                    val = dp[pos][j][c]
                    # Place a value <= j (no new comparison)
                    for v in range(1, j + 1):
                        dp[pos + 1][j][c] = (dp[pos + 1][j][c] + val) % MOD
                    # Place a value > j (new comparison, new max = v)
                    if c + 1 <= k:
                        for v in range(j + 1, m + 1):
                            dp[pos + 1][v][c + 1] = (dp[pos + 1][v][c + 1] + val) % MOD

        return sum(dp[n][j][k] for j in range(1, m + 1)) % MOD

### Go

In [ ]:
func numOfArrays(n int, m int, k int) int {
    const MOD = 1_000_000_007
    // dp[pos][max_so_far][comparisons]
    dp := make([][][]int64, n+1)
    for i := range dp {
        dp[i] = make([][]int64, m+1)
        for j := range dp[i] { dp[i][j] = make([]int64, k+1) }
    }
    // Base: empty array, max = 0, 0 comparisons — 1 way
    dp[0][0][0] = 1

    for pos := 0; pos < n; pos++ {
        for j := 0; j <= m; j++ {
            for c := 0; c <= k; c++ {
                val := dp[pos][j][c]
                if val == 0 { continue }
                // Place a value <= j (no new comparison)
                for v := 1; v <= j; v++ {
                    dp[pos+1][j][c] = (dp[pos+1][j][c] + val) % MOD
                }
                // Place a value > j (new comparison, new max = v)
                if c+1 <= k {
                    for v := j + 1; v <= m; v++ {
                        dp[pos+1][v][c+1] = (dp[pos+1][v][c+1] + val) % MOD
                    }
                }
            }
        }
    }
    var ans int64
    for j := 1; j <= m; j++ { ans = (ans + dp[n][j][k]) % MOD }
    return int(ans)
}

### Rust

In [ ]:
impl Solution {
    pub fn num_of_arrays(n: i32, m: i32, k: i32) -> i32 {
        const MOD: u64 = 1_000_000_007;
        let (n, m, k) = (n as usize, m as usize, k as usize);
        // dp[pos][max_so_far][comparisons]
        let mut dp = vec![vec![vec![0u64; k + 1]; m + 1]; n + 1];
        // Base: empty array, max = 0, 0 comparisons — 1 way
        dp[0][0][0] = 1;

        for pos in 0..n {
            for j in 0..=m {
                for c in 0..=k {
                    let val = dp[pos][j][c];
                    if val == 0 { continue; }
                    // Place a value <= j (no new comparison)
                    for v in 1..=j {
                        dp[pos + 1][j][c] = (dp[pos + 1][j][c] + val) % MOD;
                    }
                    // Place a value > j (new comparison, new max = v)
                    if c + 1 <= k {
                        for v in j + 1..=m {
                            dp[pos + 1][v][c + 1] = (dp[pos + 1][v][c + 1] + val) % MOD;
                        }
                    }
                }
            }
        }
        let ans: u64 = (1..=m).map(|j| dp[n][j][k]).sum::<u64>() % MOD;
        ans as i32
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `n=2, m=3, k=1`
Arrays of length 2 with max in [1,3], exactly 1 new-max comparison. Valid: [1,1],[2,1],[2,2],[3,1],[3,2],[3,3] (first element sets the max, second is ≤ max). Answer: **6**.

### 2. Slightly Complex
**Input:** `n=5, m=2, k=3`
Exactly 3 new-max events using values in {1,2}, length 5. The max can only increase from 1→2 once, so 3 comparisons requires value 1 to appear twice before value 2. Answer: **0** (impossible: max can only increase once with values ≤ 2).

### 3. Edge Case: Time Factor
**Input:** `n=50, m=100, k=50`
The DP table has $50 \times 100 \times 50 = 250{,}000$ states, each with up to 100 transitions — $\approx 2.5 \times 10^7$ operations. Maximum input complexity.

### 4. Edge Case: Space Factor
**Input:** `n=50, m=100, k=50`
The 3D array stores $51 \times 101 \times 51 \approx 263{,}000$ values — $\approx 2$ MB at 8 bytes each. Largest allocation for this problem.

### 5. Almost-Impossible but Plausible
**Input:** `n=1, m=1, k=1`
Only one array possible: `[1]`. Placing 1 as the first element makes exactly 1 comparison. Answer: **1**.